# Testing and debugging the dataset creation

In [ ]:
from dotenv import load_dotenv
import requests
import os
from firecrawl import Firecrawl
import json
from datetime import datetime
import re
from urllib.parse import urlsplit
import time
import sys
sys.path.append("..")
import importlib
import scripts.ingest

importlib.reload(scripts.ingest)

from scripts.ingest import (
    processed_date,
    scrape_archive_metadata,
    selection_logic,
    extract_slug,
    scrape_or_load_edition,
    filter_and_clean_parts,
    url_headline_extract,
    process_edition,
    append_and_store_edition,
    append_and_store_quiz,
    check_existence_process_append_store
)

load_dotenv("../.env")

api_key = os.getenv("FIRECRAWL_API_KEY")

client = Firecrawl(api_key=api_key)

data_path = "../data"

with open(f"{data_path}/archive.json", "r") as f:
    archive = json.load(f)

In [4]:
full_archive_metadata = scrape_archive_metadata(client)

In [64]:
# Constants and initial steps (just testing on one edition)

first_thursday = datetime.fromisoformat("2020-11-05").date()
first_worded_slug = datetime.fromisoformat("2021-09-30").date()
valid_weekdays = [3,5] # Thursday, Saturday

data_path = "../data"

index = 2284
edition_metadata = full_archive_metadata[index]

id = edition_metadata["_id"]
date = processed_date(edition_metadata)
print(date)
edition_metadata

2020-02-08


{'_id': 'oKMfr9SWcdV8hM5KapmamHWE',
 'date': '2020-02-08T10:00:00.000+00:00',
 'previewText': 'Are you ready for some football? ',
 'slug': '2020/02/08',
 'subjectLine': 'Do not pass go',
 'title': 'Are You Ready for Some Football? ',
 'websitePreviewText': None}

In [65]:
selection_logic(date)

True

In [66]:
slug = extract_slug(edition_metadata, date)
print(slug)

2020%2F02%2F08


In [67]:
edition_metadata = full_archive_metadata[index]
edition_id = edition_metadata["_id"]
iso_date = date.isoformat()

scraped_edition = scrape_or_load_edition(client, slug, edition_id, iso_date, archive)
scraped_edition

Edition for 2020-02-08 found


'[Skip to main content](https://www.morningbrew.com/issues/2020%2F02%2F08#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### ☕Do not pass go\n\nTo:Brew Readers\n\n* * *\n\n|     |\n| --- |\n| Are you ready for some football? |\n\n{beacon}\n\n| | |     |     |\n| --- | --- |\n| February 08, 2020 | [View Online](https://www.morningbrew.com/daily/issues

In [69]:
split_quiz = scraped_edition.split("<br>")
split_quiz

['[Skip to main content](https://www.morningbrew.com/issues/2020%2F02%2F08#main-content)\n\nWelcome! We use third-party tools to collect information about you as you enjoy the site. Our [Privacy Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) explains more. To opt-out of targeted ads, choose [Do Not Sell / Share My Info](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb). By proceeding, you accept our [Terms](https://www.morningbrew.com/terms-service?utm_medium=website&utm_source=terms_banner&utm_campaign=mb) and the [Policy](https://privacy.morningbrewinc.com/policies?utm_medium=website&utm_source=terms_banner&utm_campaign=mb).\n\nGot it\n\n###### ☕Do not pass go\n\nTo:Brew Readers\n\n* * *\n\n|     |\n| --- |\n| Are you ready for some football? |\n\n{beacon}\n\n| | |     |     |\n| --- | --- |\n| February 08, 2020 | [View Online](https://www.morningbrew.com/daily/issue

In [83]:
questions_and_answers = filter_and_clean_parts(scraped_edition)
questions_and_answers

['1. “Antarctica logs [hottest temperature on record](https://www.theguardian.com/world/2020/feb/07/antarctica-logs-hottest-temperature-on-record-with-a-reading-of-183c?utm_source=morning_brew) with a reading of 18.3C” (65 degrees Fahrenheit)',
 '2. “[Paper crumbs](https://media.giphy.com/media/l2Jeal95KEUTLzNpS/giphy.gif?utm_source=morning_brew) of Nancy Pelosi’s State of the Union address wind up on eBay for $1.5 million”\xa0',
 '3. “Flyers mascot Gritty [cleared of any wrongdoing](https://www.espn.com/nhl/story/_/id/28626947/flyers-mascot-cleared-wrongdoing-alleged-assault-case?utm_source=morning_brew) in alleged assault” ',
 '4. “[Iowa caucuses](https://nypost.com/2020/02/03/iowa-caucuses-attendee-drops-breaks-concealed-bottle-of-wine-at-polling-site/?utm_source=morning_brew): Attendee drops, breaks ‘concealed bottle of wine’” ',
 'Not a shred of the speech is on eBay as far as we know.']

In [87]:
url = re.compile(r'(\(.*?\))').search(questions_and_answers[0]).group().strip("()")
questions_and_answers[0].replace(url, "").replace("()", "").replace("[", "").replace("]", "").lstrip("1234. ")

'“Antarctica logs hottest temperature on record with a reading of 18.3C” (65 degrees Fahrenheit)'

In [ ]:
questions_and_answers = clean_q4_and_answer(questions_and_answers)
questions_and_answers

In [88]:
# quiz = {
#         "_id": edition_id,
#         "date": date.isoformat(),
#         "source_url": f"https://www.morningbrew.com/issues/{slug}",
#         "headlines": [],
#         "answer": questions_and_answers[4]
#     }

for index in range(4):
    if index < 4:
        headline, url = url_headline_extract(index, questions_and_answers)
        # quiz["headlines"].append({"text": headline, "url": url})
        print(headline, url)#quiz["headlines"].append({"text": headline, "url": url})

# quiz

Antarctica logs hottest temperature on record with a reading of 18.3C” (65 degrees Fahrenheit) https://www.theguardian.com/world/2020/feb/07/antarctica-logs-hottest-temperature-on-record-with-a-reading-of-183c
Paper crumbs of Nancy Pelosi’s State of the Union address wind up on eBay for $1.5 million”  https://media.giphy.com/media/l2Jeal95KEUTLzNpS/giphy.gif
Flyers mascot Gritty cleared of any wrongdoing in alleged assault”  https://www.espn.com/nhl/story/_/id/28626947/flyers-mascot-cleared-wrongdoing-alleged-assault-case
Iowa caucuses: Attendee drops, breaks ‘concealed bottle of wine’”  https://nypost.com/2020/02/03/iowa-caucuses-attendee-drops-breaks-concealed-bottle-of-wine-at-polling-site/


In [89]:
process_edition(index, full_archive_metadata, date, archive, iso_date)

Edition for 2020-02-08 found
Extracting headline text and URLs
{
  "_id": "X27WcffkRFi9NhntQFjrPR",
  "date": "2020-02-08",
  "source_url": "https://www.morningbrew.com/issues/your-best-shot",
  "headlines": [
    {
      "text": "Antarctica logs hottest temperature on record with a reading of 18.3C\u201d (65 degrees Fahrenheit)",
      "url": "https://www.theguardian.com/world/2020/feb/07/antarctica-logs-hottest-temperature-on-record-with-a-reading-of-183c"
    },
    {
      "text": "Paper crumbs of Nancy Pelosi\u2019s State of the Union address wind up on eBay for $1.5 million\u201d\u00a0",
      "url": "https://media.giphy.com/media/l2Jeal95KEUTLzNpS/giphy.gif"
    },
    {
      "text": "Flyers mascot Gritty cleared of any wrongdoing in alleged assault\u201d ",
      "url": "https://www.espn.com/nhl/story/_/id/28626947/flyers-mascot-cleared-wrongdoing-alleged-assault-case"
    },
    {
      "text": "Iowa caucuses: Attendee drops, breaks \u2018concealed bottle of wine\u2019\u201d 

{'_id': 'X27WcffkRFi9NhntQFjrPR',
 'date': '2020-02-08',
 'source_url': 'https://www.morningbrew.com/issues/your-best-shot',
 'headlines': [{'text': 'Antarctica logs hottest temperature on record with a reading of 18.3C” (65 degrees Fahrenheit)',
   'url': 'https://www.theguardian.com/world/2020/feb/07/antarctica-logs-hottest-temperature-on-record-with-a-reading-of-183c'},
  {'text': 'Paper crumbs of Nancy Pelosi’s State of the Union address wind up on eBay for $1.5 million”\xa0',
   'url': 'https://media.giphy.com/media/l2Jeal95KEUTLzNpS/giphy.gif'},
  {'text': 'Flyers mascot Gritty cleared of any wrongdoing in alleged assault” ',
   'url': 'https://www.espn.com/nhl/story/_/id/28626947/flyers-mascot-cleared-wrongdoing-alleged-assault-case'},
  {'text': 'Iowa caucuses: Attendee drops, breaks ‘concealed bottle of wine’” ',
   'url': 'https://nypost.com/2020/02/03/iowa-caucuses-attendee-drops-breaks-concealed-bottle-of-wine-at-polling-site/'}],
 'answer': 'Not a shred of the speech is on

In [16]:
q_search = re.compile(r'[1234]\. (“)?.*\[([\w\s\.\-])*\](\(.*\))')
part = '# [Demon Stores Look for New Hosts](https://www.morningbrew.com/daily/stories/2020/10/30/halloween-popup-shops-mixed-success-2020?email={{profile.email}}) |\n\n![Spirit Halloween store](https://dlp31coh2a67q.cloudfront.net/eyJrZXkiOiJ1cGxvYWRzL21lZGl1bS9hc3NldC8yOTkwL3NwaXJpdF9oYWxsb3dlZW5fc3RvcmUuanBnIiwiYnVja2V0Ijoib3Nsby1wcm9kdWN0aW9uIiwiZWRpdHMiOnsicmVzaXplIjp7IndpZHRoIjoxMDAwLCJoZWlnaHQiOm51bGx9fX0=)\n\n_Ben McCanna/Portland Portland Press Herald via Getty Images_\n\n|     |\n| --- |\n| Every year, pop-up Halloween stores rise from the underworld to possess empty storefronts vacated by mortal retailers.\n\nTypically, this non-traditional leasing model means seasonal pop-ups perform well when the rest of retail flounders. But even with 2020 laying out a feast of fresh retail corpses, are spooky stores immune to the pandemic? Let\'s check in with two retail reanimaters:\n\n1. **Spirit Halloween** is the host with the most. It actually opened [more stores](https://www.nytimes.com/2020/10/12/style/spirit-halloween-costume-store-undead-forever.html?utm_source=morning_brew) this year (1,400+) than last year, and its CEO said last month the company is "anticipating a Halloween on par with last year."\n2. **Party City** was far less bullish, opening 91% fewer of its Halloween City locations this year. With just 25 stores across the U.S., these days it\'s more like a Halloween Town.\n\n**Zoom out:** Halloween City was showing zombie-like symptoms even before the pandemic. Last year, sales per store declined nearly [21% annually](https://www.retaildive.com/news/party-citys-october-sales-a-disaster-with-halloween-stores-down-21/566955/?utm_source=morning_brew)—a major problem for parent Party City considering Halloween sales account for a fifth of its total U.S. retail revenue.\n\n|     |\n| --- |\n| [![](https://media.sailthru.com/5z8/1k4/4/2/5e865734bf6f7.jpg)](https://www.facebook.com/sharer/sharer.php?u=https%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fstories%2F2020%2F10%2F30%2Fhalloween-popup-shops-mixed-success-2020)[![](https://media.sailthru.com/5z8/1k4/4/2/5e86574d1eb82.jpg)](https://twitter.com/intent/tweet?text=Party+City%27s+opening+91%25+fewer+of+its+Halloween+City+locations+this+year&url=https://www.morningbrew.com/daily/stories/2020/10/30/halloween-popup-shops-mixed-success-2020&via=MorningBrew)[![](https://media.sailthru.com/5z8/1k4/4/2/5e86576e04837.jpg)](https://www.linkedin.com/shareArticle?mini=true&url=https%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fstories%2F2020%2F10%2F30%2Fhalloween-popup-shops-mixed-success-2020&title=Halloween+Pop-Up+Shops+Have+Mixed+Success+In+2020&source=www.morningbrew.com&summary=Party+City%27s+opening+91%25+fewer+of+its+Halloween+City+locations+this+year)[![](https://media.sailthru.com/5z8/1k4/4/2/5e8657887c532.jpg)](mailto:?subject=Check%20out%20this%20story%20from%20Morning%20Brew%21&body=Halloween%20Pop-Up%20Shops%20Have%20Mixed%20Success%20In%202020%3A%20https%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fstories%2F2020%2F10%2F30%2Fhalloween-popup-shops-mixed-success-2020%0A%0AWant%20more%20great%20content%3F%20Subscribe%20to%20Daily%20Brew%20%E2%80%94%20Delivering%20the%20latest%20business%20news%20from%20Wall%20St.%20to%20Silicon%20Valley%2C%20daily.%0Ahttps%3A%2F%2Fwww.morningbrew.com%2Fdaily%2Fr%2F%3Fkid%3D{{profile.vars.referral_code}}%26utm_source%3Demail_share%0A) | | |\n| --- |\n\n| |     |\n| --- |\n| ### FOOD'
matched = q_search.search(part)
matched.group()

'1. **Spirit Halloween** is the host with the most. It actually opened [more stores](https://www.nytimes.com/2020/10/12/style/spirit-halloween-costume-store-undead-forever.html?utm_source=morning_brew) this year (1,400+)'

In [90]:
with open(f"{data_path}/quizzes.json", "r") as f:
    quizzes = json.load(f)
print(f"Number of quizzes: {len(quizzes)}")

with open(f"{data_path}/exceptions.json", "r") as f:
    exceptions = json.load(f)
print(f"Number of exceptions: {len(exceptions)}")


Number of quizzes: 336


FileNotFoundError: [Errno 2] No such file or directory: '../data/exceptions.json'